# Logistic Regression Model for Heart Disease Prediction

## Overview
This notebook builds a complete Logistic Regression model to predict whether a person has heart disease using the Cardiovascular Disease Dataset.

### Target Variable
- **1**: Patient has heart disease
- **0**: Patient does not have heart disease

### Machine Learning Pipeline
1. Data Understanding & Exploration
2. Data Cleaning
3. Exploratory Data Analysis (EDA)
4. Feature Selection
5. Outlier Detection & Handling
6. Data Preprocessing
7. Train-Test Split
8. Model Training
9. Overfitting Check
10. Model Evaluation
11. Model Saving
12. Prediction on New Data

## Section 1: Import Required Libraries

Import all necessary libraries for data processing, visualization, and machine learning model building.


In [2]:
# Data Processing Libraries
import pandas as pd
import numpy as np

# Data Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, roc_curve, auc)

# Model Saving Libraries
import pickle
import joblib
import os

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print(" All required libraries imported successfully!")

 All required libraries imported successfully!


## Section 2: Load and Explore Dataset

Load the CSV file from the folder structure into a pandas DataFrame and perform initial exploration.

In [3]:
# Load the CSV file
csv_path = 'Cardiovascular_Disease_Dataset.csv'

# Check if file exists
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f" Data loaded successfully!")
    print(f"File: {csv_path}\n")
else:
    raise FileNotFoundError(f"CSV file not found at {csv_path}")

# Display dataset shape and basic information
print("=" * 80)
print("DATASET SHAPE AND BASIC INFORMATION")
print("=" * 80)
print(f"Dataset shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")

# Display column names and data types
print("Column Names and Data Types:")
print(df.dtypes)
print("\n")

# Display first 5 rows
print("=" * 80)
print("FIRST 5 ROWS OF THE DATASET")
print("=" * 80)
print(df.head())
print()

# ============================================================================
# REMOVE UNWANTED COLUMNS (IDs and non-predictive features)
# ============================================================================
print("=" * 80)
print("REMOVING UNWANTED COLUMNS")
print("=" * 80)

# Define unwanted columns (IDs, identifiers)
unwanted_col_patterns = ['id', 'ID', 'patient', 'patientid', 'record', 'index']

# Find unwanted columns
columns_to_remove = []
for col in df.columns:
    col_lower = col.lower()
    for pattern in unwanted_col_patterns:
        if pattern in col_lower:
            columns_to_remove.append(col)
            break

if columns_to_remove:
    print(f"\nColumns identified for removal (IDs and identifiers):")
    for col in columns_to_remove:
        print(f"  - {col}")
    
    df.drop(columns=columns_to_remove, inplace=True)
    print(f"\n✓ Removed {len(columns_to_remove)} unwanted columns")
else:
    print("\n✓ No unwanted columns found")

print(f"\nDataset shape after removing unwanted columns: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")
print()

# ============================================================================
# REMOVE UNWANTED COLUMNS (IDs and non-predictive features)
# ============================================================================
print("=" * 80)
print("REMOVING UNWANTED COLUMNS")
print("=" * 80)

# Define unwanted columns (IDs, identifiers)
unwanted_col_patterns = ['id', 'ID', 'patient', 'patientid', 'record', 'index']

# Find unwanted columns
columns_to_remove = []
for col in df.columns:
    col_lower = col.lower()
    for pattern in unwanted_col_patterns:
        if pattern in col_lower:
            columns_to_remove.append(col)
            break

if columns_to_remove:
    print(f"\nColumnsIdentified for removal (IDs and identifiers):")
    for col in columns_to_remove:
        print(f"  - {col}")
    
    df.drop(columns=columns_to_remove, inplace=True)
    print(f"\n✓ Removed {len(columns_to_remove)} unwanted columns")
else:
    print("\n✓ No unwanted columns found")

print(f"\nDataset shape after removing unwanted columns: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")

 Data loaded successfully!
File: Cardiovascular_Disease_Dataset.csv

DATASET SHAPE AND BASIC INFORMATION
Dataset shape: (1000, 14)
Rows: 1000, Columns: 14

Column Names and Data Types:
patientid              int64
age                    int64
gender                 int64
chestpain              int64
restingBP              int64
serumcholestrol        int64
fastingbloodsugar      int64
restingrelectro        int64
maxheartrate           int64
exerciseangia          int64
oldpeak              float64
slope                  int64
noofmajorvessels       int64
target                 int64
dtype: object


FIRST 5 ROWS OF THE DATASET
   patientid  age  gender  chestpain  restingBP  serumcholestrol  \
0     103368   53       1          2        171                0   
1     119250   40       1          0         94              229   
2     119372   49       1          2        133              142   
3     132514   43       1          0        138              295   
4     146211   31       1

In [4]:
# Check for missing values
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': missing_values.values,
    'Percentage': missing_percentage.values
})
print(missing_df[missing_df['Missing_Count'] > 0])
if missing_df['Missing_Count'].sum() == 0:
    print(" No missing values found in the dataset!")
print("\n")

# Display basic statistics
print("=" * 80)
print("BASIC STATISTICS")
print("=" * 80)
print(df.describe())

MISSING VALUES ANALYSIS
Empty DataFrame
Columns: [Column, Missing_Count, Percentage]
Index: []
 No missing values found in the dataset!


BASIC STATISTICS
              age       gender    chestpain    restingBP  serumcholestrol  \
count  1000.00000  1000.000000  1000.000000  1000.000000      1000.000000   
mean     49.24200     0.765000     0.980000   151.747000       311.447000   
std      17.86473     0.424211     0.953157    29.965228       132.443801   
min      20.00000     0.000000     0.000000    94.000000         0.000000   
25%      34.00000     1.000000     0.000000   129.000000       235.750000   
50%      49.00000     1.000000     1.000000   147.000000       318.000000   
75%      64.25000     1.000000     2.000000   181.000000       404.250000   
max      80.00000     1.000000     3.000000   200.000000       602.000000   

       fastingbloodsugar  restingrelectro  maxheartrate  exerciseangia  \
count        1000.000000      1000.000000   1000.000000    1000.000000   
mea

In [5]:
print("=" * 80)
print("REMOVING UNWANTED COLUMNS (IDs, Identifiers)")
print("=" * 80)

# Check for common unwanted column names that should not be used for prediction
unwanted_patterns = ['id', 'ID', 'patient', 'patientid', 'record', 'index']

# Find columns to remove
columns_to_remove = []
for col in df.columns:
    col_lower = col.lower()
    for pattern in unwanted_patterns:
        if pattern in col_lower:
            columns_to_remove.append(col)
            break

# Remove unwanted columns
if columns_to_remove:
    print(f"\nUnwanted columns found (not useful for prediction):")
    for col in columns_to_remove:
        print(f"   '{col}' - will be removed")
    
    df.drop(columns=columns_to_remove, inplace=True)
    print(f"\n Removed {len(columns_to_remove)} unwanted columns")
else:
    print("\n No unwanted columns (like patientid) found")

print(f"\nDataset shape after cleanup: {df.shape}")
print(f"Useful columns for training: {list(df.columns)}")
print()

REMOVING UNWANTED COLUMNS (IDs, Identifiers)

 No unwanted columns (like patientid) found

Dataset shape after cleanup: (1000, 13)
Useful columns for training: ['age', 'gender', 'chestpain', 'restingBP', 'serumcholestrol', 'fastingbloodsugar', 'restingrelectro', 'maxheartrate', 'exerciseangia', 'oldpeak', 'slope', 'noofmajorvessels', 'target']

